In [1]:
# [COLAB SETUP]
import sys
import os

if "google.colab" in sys.modules:
    print("Running in Google Colab. Setting up environment...")
    
    # Mount Google Drive to persist the datasets and cloned repository
    from google.colab import drive
    drive.mount('/content/drive')
    
    repo_path = '/content/drive/MyDrive/Sinhala-Script-Language-Identification-LangID-for-Sinhala-Pali-and-Sanskrit'
    
    if not os.path.exists(repo_path):
        print(f"Cloning repository into {repo_path}...")
        os.makedirs('/content/drive/MyDrive', exist_ok=True)
        os.system(f'git clone https://github.com/Maleesha-K/Sinhala-Script-Language-Identification-LangID-for-Sinhala-Pali-and-Sanskrit.git {repo_path}')
        
    os.chdir(repo_path + '/data_pipeline')
    print("Installing base dependencies...")
    os.system('pip install -q pandas scikit-learn transformers torch tqdm peft')
    print("Setup complete!")


In [2]:
# NOTE: If running this notebook manually in the IDE, make sure you have transformers and torch installed!
input_dir = 'datasets/preprocessed'
output_dir = 'datasets/benchmark_finetuned_results'
model_path = 'models/finetuned/xlm-roberta-base-langid'


In [3]:
import os
# Auto-resolve the project root if running manually
if not os.path.exists("Makefile") and os.path.exists("../../Makefile"):
    os.chdir("../../")

import json
import glob
import pandas as pd
from sklearn.metrics import accuracy_score, classification_report, f1_score

TARGET_LANGUAGES = {
    "eng": "en",  # English
    "sin": "si",  # Sinhala 
    "san": "sa",  # Sanskrit
    "tam": "ta",  # Tamil
    "hin": "hi",  # Hindi
    "ben": "bn",  # Bengali
    "arb": "ar",  # Arabic
    "fra": "fr",  # French
    "deu": "de",  # German
    "pli": "pi",  # Pali 
}

def load_dataset(file_path):
    print(f"\nLoading {os.path.basename(file_path)}...")
    records = []
    with open(file_path, encoding="utf-8") as f:
        for line in f:
            row = json.loads(line)
            if row.get("label") in TARGET_LANGUAGES:
                records.append(row)
    df = pd.DataFrame(records)
    if not df.empty:
        df["mapped_label"] = df["label"].map(TARGET_LANGUAGES)
        print(f"Loaded {len(df)} rows across {df['label'].nunique()} target languages")
    else:
        print("No matching target languages found in this dataset.")
    return df

def evaluate_and_save(results, model_name, dataset_name, target_labels):
    acc = accuracy_score(results["true_label"], results["predicted_label"])
    
    # We use zero_division=0 to suppress warnings when a language isn't predicted
    macro_f1 = f1_score(
        results["true_label"], results["predicted_label"],
        average="macro", labels=target_labels, zero_division=0
    )

    print("\n" + "=" * 48)
    print(f"FINETUNED BENCHMARK RESULTS ({model_name} on {dataset_name})")
    print("=" * 48)
    print(f"Accuracy:  {acc * 100:.2f}%")
    print(f"Macro F1:  {macro_f1 * 100:.2f}%")
    print("=" * 48)
    print("\nPer-language breakdown:\n")
    print(classification_report(
        results["true_label"], results["predicted_label"],
        labels=target_labels, digits=4, zero_division=0
    ))

    os.makedirs(output_dir, exist_ok=True)
    safe_model_name = os.path.basename(model_name).replace('/', '_').replace(' ', '_').replace('-', '_').lower()
    out_file = os.path.join(output_dir, f"{safe_model_name}_{dataset_name}.csv")
    results.to_csv(out_file, index=False)
    print(f"\nSaved predictions to {out_file}\n")
    return results

dataset_files = glob.glob(os.path.join(input_dir, "*.jsonl"))
if not dataset_files:
    print(f"No datasets found in {input_dir}.")


In [4]:
from transformers import pipeline, AutoModelForSequenceClassification, AutoTokenizer, AutoConfig
from tqdm.auto import tqdm
import torch

device = 0 if torch.cuda.is_available() else -1

print(f"Loading Finetuned XLM-RoBERTa language detection model from {model_path}...")
import torch.distributed.tensor  # Fix peft bug
config = AutoConfig.from_pretrained(model_path)
model = AutoModelForSequenceClassification.from_pretrained(model_path, config=config, ignore_mismatched_sizes=True)
tokenizer = AutoTokenizer.from_pretrained(model_path)

pipe = pipeline("text-classification", model=model, tokenizer=tokenizer, device=device)

target_labels = sorted(set(TARGET_LANGUAGES.values()))

for file_path in dataset_files:
    dataset_name = os.path.splitext(os.path.basename(file_path))[0]
    df = load_dataset(file_path)
    if df.empty: continue
    
    texts = df["text"].astype(str).tolist()
    print(f"Evaluating {len(texts)} samples with finetuned model...")
    
    # Run predictions in batches
    preds = pipe(texts, batch_size=32, truncation=True, max_length=512)
    predicted_labels = [p['label'] for p in preds]

    results = df[["text", "label", "source"]].copy()
    results["true_label"] = df["mapped_label"]
    results["predicted_label"] = predicted_labels

    evaluate_and_save(results, model_path, dataset_name, target_labels)


/home/vihanga/Desktop/programming/Sinhala-Script-Language-Identification-LangID-for-Sinhala-Pali-and-Sanskrit/data_pipeline/models/benchmark/ConLID/.venv/lib/python3.10/site-packages/tqdm/auto.py:21: TqdmWarning: IProgress not found. Please update jupyter and ipywidgets. See https://ipywidgets.readthedocs.io/en/stable/user_install.html
  from .autonotebook import tqdm as notebook_tqdm


Loading Finetuned XLM-RoBERTa language detection model from models/finetuned/xlm-roberta-base-langid...


Some weights of XLMRobertaForSequenceClassification were not initialized from the model checkpoint at papluca/xlm-roberta-base-language-detection and are newly initialized because the shapes did not match:
- classifier.out_proj.bias: found shape torch.Size([20]) in the checkpoint and torch.Size([25]) in the model instantiated
- classifier.out_proj.weight: found shape torch.Size([20, 768]) in the checkpoint and torch.Size([25, 768]) in the model instantiated
You should probably TRAIN this model on a down-stream task to be able to use it for predictions and inference.



Loading flores_plus.jsonl...
Loaded 15143 rows across 10 target languages
Evaluating 15143 samples with finetuned model...

FINETUNED BENCHMARK RESULTS (models/finetuned/xlm-roberta-base-langid on flores_plus)
Accuracy:  93.14%
Macro F1:  92.67%

Per-language breakdown:

              precision    recall  f1-score   support

          ar     1.0000    0.5000    0.6667      2024
          bn     0.9941    1.0000    0.9970      1012
          de     1.0000    1.0000    1.0000      1012
          en     0.6305    1.0000    0.7734      1012
          fr     0.7556    0.9990    0.8604      1012
          hi     1.0000    0.9990    0.9995      1012
          pi     0.9763    0.9927    0.9844      3027
          sa     0.9970    0.9977    0.9974      1327
          si     0.9934    1.0000    0.9967      2693
          ta     0.9825    1.0000    0.9912      1012

    accuracy                         0.9314     15143
   macro avg     0.9329    0.9488    0.9267     15143
weighted avg     0.9512